#### Basic Agent Implementation with Tools using LangChain

##### Overview
This notebook demonstrates advanced AI agent creation using LangChain with tool integration, including:
- **Tool Creation**: Building custom tools from functions
- **Web Search Integration**: DuckDuckGo search for real-time information
- **Chain Tools**: Wrapping LLM chains as tools for agents
- **Agent Orchestration**: Zero-shot agents with reasoning capabilities
- **Error Handling**: Managing tool failures and fallback strategies

##### Key Components
1. **Custom Tools**: Converting Python functions into LangChain tools
2. **Web Search Tools**: DuckDuckGo integration for external data
3. **Chain Tools**: LLM chains wrapped as tools for agents
4. **Agent Initialization**: Setting up agents with multiple tools
- **Tool Selection**: Agents choosing appropriate tools for tasks

##### Use Cases Demonstrated
- Weather information retrieval
- Web content summarization
- Sports advice generation
- Multi-step problem solving

##### Agent Behavior Analysis
- **Tool Selection Logic**: How agents choose between tools
- **Fallback Strategies**: Handling search failures
- **Hallucination Issues**: When agents create fake information
- **Error Recovery**: Managing tool unavailability

In [6]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType

In [7]:
import requests
from bs4 import BeautifulSoup

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables (e.g., OpenAI API keys) from a .env file located on the local system
env_path = r'/Users/nareshchaurasia/nc/Python-TechHub/AI-ML-GenAI/Agentic-AI/.env'
load_dotenv(env_path)

True

In [9]:
import warnings
warnings.filterwarnings('ignore')

### Agent Creation

* Agent Creation using Tools
* **Initial Choice**: The agent correctly identified it needed weather information and chose duckduckgo_search first
* **Search Failure**: DuckDuckGo returned "No good DuckDuckGo Search Result was found" - this is a common issue with the search tool
* **Fallback Strategy**: When the search failed, the agent switched to your simple_tool
* **Tool Limitation**: The simple_tool just returns the input text, so it's not actually helpful
* **Hallucination**: The agent then made up weather data ("65°F with scattered showers") without any real information

In [ ]:
# Creating a LangChain Agent with Tools
# This cell demonstrates how to create an AI agent that can use multiple tools
# including a custom function tool and web search to answer questions

from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a simple tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"  # Return formatted response

#creating tool from function
my_tool = Tool.from_function(func=my_tool_function, name="simple_tool", description="A simple tool")  # Convert function to LangChain tool
ddg_search = DuckDuckGoSearchRun()  # Initialize web search tool

# Initialize the LLM and the agent
llm = ChatOpenAI(model="gpt-3.5-turbo")  # Create LLM instance
tools = [ddg_search,my_tool]  # Combine available tools

# Create an agent
agent = initialize_agent(tools, llm, agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)  # Setup zero-shot agent with reasoning

# Use the agent
response = agent.run("What's the weather like today in London?")  # Execute agent task
print(response)  # Output: The agent will choose to use the tool or model to respond based on context

### Agent Creation using Tools

* Agent Creation using Tools
* Creates a prompt template for summarizing content
* Initializes a ChatGPT model (16k context window)
* Builds an LLM chain that combines the prompt and model
* Wraps the chain in a tool that agents can use

In [ ]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-3.5-turbo-16k")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [ ]:
tools = [ddg_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

**Agent's Decision Flow**

**Initial Strategy**

* Agent suggests looking for professional tips

**First Search Attempt**

* Uses `duckduckgo_search` with query:
  `"tips to become a good footballer"`
* **Result:** ❌ Fails

**Second Search Attempt**

* Tries a more specific search with BBC site restriction
* **Result:** ❌ Fails

**Tool Switch**

* Abandons search
* Switches to **Summarizer tool**

**Summarizer Usage**

* Provides a fake URL:
  `"URL of an article about becoming a good footballer"`

**Generic Response**

* Summarizer returns a **generic summary** (no real article accessed)

**Final Answer**

* Agent provides **vague advice** based on the fake summary

---

**Key Issues**

**Search Failures**

* DuckDuckGo search consistently returns:
  `"No good DuckDuckGo Search Result was found"`
* Same technical issue observed earlier

**Agent's Logic Problem**

* When search fails:

  * Agent creates a **fake URL** for the Summarizer
* Summarizer limitation:

  * Cannot access real web content
  * Generates **generic response**
* Critical issue:

  * Agent treats this **fake summary as real information**

In [ ]:
prompt = """Please tell me how to become a good footballer"""
print(agent.invoke(prompt))